In [1]:
import pandas as pd
from pathlib import Path


def err(msg):
    print('Error: ' + msg)
    exit(0)
    
def loadNormData(path):
    rawData = []
    goldData = []
    curSent = []

    for line in open(path):
        tok = line.strip().split('\t')

        if tok == [''] or tok == []:
            rawData.append([x[0] for x in curSent])
            goldData.append([x[1] for x in curSent])
            curSent = []

        else:
            if len(tok) > 2:
                err('erroneous input, line:\n' + line + '\n in file ' + path + ' contains more then two elements')
            if len(tok) == 1:
                tok.append('')
            curSent.append(tok)

    # in case file does not end with newline
    if curSent != []:
        rawData.append([x[0] for x in curSent])
        goldData.append([x[1] for x in curSent])
    return rawData, goldData


datasets = {}
for p in Path("./datasets").rglob("*"):
    if p.is_file():
        name = f"{p.parent.name}-{p.name.replace('.norm', '')}"
        raw, norm = loadNormData(p)
        datasets[name] = [{"raw": r, "norm": n} for r,n in list(zip(raw, norm))]

pd.DataFrame(datasets['en-dev']).head(3)

,raw,norm
0,"[@cdutra5, bruh, get, out, yo, feelings, lol]","[@cdutra5, brother, get, out, your, feelings, ..."
1,"[rt, @demberel_s, :, manan, dund, xaragdax, te...","[rt, @demberel_s, :, manan, dund, xaragdax, te..."
2,"[why, dese, niggas, think, dey, doin, summn]","[why, these, niggers, think, they, doing, some..."


In [2]:
from datasets import Dataset, DatasetDict, concatenate_datasets

splits = {"train": [], "validation": [], "test": []}

for k, ds in datasets.items():
    lang, split = k.split("-", 1)  # e.g. en-train, de-dev
    split = "validation" if split == "dev" else split

    if split in splits:
        ds = Dataset.from_list(ds)
        ds = ds.add_column("lang", [lang] * len(ds))
        splits[split].append(ds)

out = DatasetDict({
    s: concatenate_datasets(dsets)
    for s, dsets in splits.items()
    if dsets
})

In [3]:
datasets.keys()

dict_keys(['da-test', 'da-train', 'de-dev', 'de-test', 'de-train', 'en-dev', 'en-test', 'en-train', 'es-test', 'es-train', 'hr-dev', 'hr-test', 'hr-train', 'id-dev', 'id-test', 'id-train', 'iden-dev', 'iden-test', 'iden-train', 'it-test', 'it-train', 'ja-dev', 'ja-test', 'ja-train', 'nl-dev', 'nl-test', 'nl-train', 'sl-dev', 'sl-test', 'sl-train', 'sr-dev', 'sr-test', 'sr-train', 'th-dev', 'th-test', 'th-train', 'tr-test', 'tr-train', 'trde-test', 'trde-train', 'vi-dev', 'vi-test', 'vi-train', 'ko-dev', 'ko-test', 'ko-train'])

## Upload data

In [4]:
# DatasetDict({
#     "train": out["train"],
#     "validation": out["validation"],
# }).push_to_hub("weerayut/multilexnorm2026-pub", private=True)

In [5]:
# DatasetDict({
#     "test": out["test"],
# }).push_to_hub("weerayut/multilexnorm2026-test", private=True)

## Data

In [6]:
from datasets import load_dataset

pub_data = load_dataset("weerayut/multilexnorm2026-pub")
test_data = load_dataset("weerayut/multilexnorm2026-test")

pub_data, test_data

(DatasetDict({
     train: Dataset({
         features: ['raw', 'norm', 'lang'],
         num_rows: 39178
     })
     validation: Dataset({
         features: ['raw', 'norm', 'lang'],
         num_rows: 8408
     })
 }),
 DatasetDict({
     test: Dataset({
         features: ['raw', 'norm', 'lang'],
         num_rows: 11956
     })
 }))

In [7]:
lang = "en"
en_train = pub_data["train"].filter(lambda x: x["lang"] == lang)
en_validation = pub_data["validation"].filter(lambda x: x["lang"] == lang)

pd.DataFrame(en_train).head(8)

,raw,norm,lang
0,"[rt, @teddyferrari1, :, "", ah, ..., @datzmenon...","[rt, @teddyferrari1, :, "", ah, ..., @datzmenon...",en
1,"[u, have, a, very, sexy, header, @jaibrooks1, ...","[you, have, a, very, sexy, header, @jaibrooks1...",en
2,"[i, miss, u, my, bie, !, where, u, wanna, out,...","[i, miss, you, my, bie, !, where, you, want to...",en
3,"["", cantik, ., rt, @historyinpics, :, julie, c...","["", cantik, ., rt, @historyinpics, :, julie, c...",en
4,"[rt, @fvckxhemmings, :, did, calum, slip, ?!!,...","[rt, @fvckxhemmings, :, did, calum, slip, ?!!,...",en
5,"[@daviddarkshade, and, you, didnt, make, this,...","[@daviddarkshade, and, you, didn't, make, this...",en
6,"[@xoxoxyareli, @billmillerprobz, pshh, i, went...","[@xoxoxyareli, @billmillerprobz, pshh, i, went...",en
7,"[@tindency, syempre, in, the, right, age, hahaha]","[@tindency, syempre, in, the, right, age, hahaha]",en


## Stats

In [8]:
from collections import defaultdict

newlangs = ['th', 'vi', 'ja', 'ko', 'id']
code2lang = {'th': 'Thai',
 'vi': 'Vietnamese',
 'id': 'Indonesian',
 'ja': 'Japanese',
 'ko': 'Korean',
 'hr': 'Croatian',
 'da': 'Danish',
 'nl': 'Dutch',
 'en': 'English',
 'de': 'German',
 'iden': 'Indonesian-English',
 'it': 'Italian',
 'sr': 'Serbian',
 'sl': 'Slovenian',
 'es': 'Spanish',
 'tr': 'Turkish',
 'trde': 'Turkish-German'}


def count_tokens_by_lang(dataset, split_name):
    counts = defaultdict(int)

    for ex in dataset[split_name]:
        counts[f'{ex["lang"]}'] += len(ex["raw"])

    return counts

stats = {}
stats["train"] = count_tokens_by_lang(pub_data, "train")
stats["validation"] = count_tokens_by_lang(pub_data, "validation")
stats["test"] = count_tokens_by_lang(test_data, "test")

stats = pd.DataFrame(stats)
stats['total'] = stats.sum(axis=1)
stats.sort_index()
stats = stats.reset_index().rename(columns={"index": "lang"})

In [9]:
from datasets import concatenate_datasets

def count_tokens_by_lang(dataset):
    counts = defaultdict(int)
    for ex in dataset:
        counts[f'{code2lang[ex["lang"]]}: {ex["lang"]}'] += len(ex["raw"])
    return counts

merged = concatenate_datasets([pub_data["train"], pub_data["validation"], test_data["test"]])
counts = count_tokens_by_lang(merged)
print(counts)

defaultdict(<class 'int'>, {'Danish: da': 20206, 'German: de': 24948, 'English: en': 73806, 'Spanish: es': 13824, 'Croatian: hr': 89052, 'Indonesian: id': 48716, 'Indonesian-English: iden': 23124, 'Italian: it': 14641, 'Japanese: ja': 95416, 'Dutch: nl': 21657, 'Slovenian: sl': 75276, 'Serbian: sr': 91738, 'Thai: th': 200915, 'Turkish: tr': 8082, 'Turkish-German: trde': 16508, 'Vietnamese: vi': 128685, 'Korean: ko': 16577})


In [10]:
from utils import counting, mfr, evaluate

def mfrs(train, lang):
    # print(f"\nLang: {lang}")
    train = train.filter(lambda x: x["lang"] == lang)
    test = test_data["test"].filter(lambda x: x["lang"] == lang)
    counts = counting(train)

    ds = pd.DataFrame(test)
    ds['pred'] = ds['raw'].apply(lambda x: mfr(x, counts))
    lai, acc, err = evaluate(ds['raw'].tolist(), ds['norm'].tolist(), ds['pred'].tolist(), info=False)
    
    return round(err*100, 2)

# Smoke test
print(mfrs(pub_data["train"], 'en'))

train = concatenate_datasets([pub_data["train"], pub_data["validation"]])
stats['mfr'] = stats['lang'].apply(lambda x: mfrs(train, x))

64.93


In [11]:
stats

,lang,train,validation,test,total,mfr
0,da,16448,NaN,3758,20206.0,49.68
1,de,15006,4860.0,5082,24948.0,34.35
2,en,35216,9169.0,29421,73806.0,66.57
3,es,7189,NaN,6635,13824.0,25.57
4,hr,54416,18941.0,15695,89052.0,41.53
5,id,35502,4306.0,8908,48716.0,59.75
6,iden,13949,4809.0,4366,23124.0,61.51
7,it,12645,NaN,1996,14641.0,16.83
8,ja,61903,10919.0,22594,95416.0,6.32
9,nl,12381,3863.0,5413,21657.0,39.39


In [12]:
def norm_ratio(data, language):
    data = data.filter(lambda x: x["lang"] == language)

    count, not_norm = 0, 0
    for items in data:
        count += len(items['raw'])
        not_norm += sum([1 for x in zip(items['raw'], items['norm']) if x[0]==x[1]])

    ratio = round((1-(not_norm/count))*100, 2)
    print(f"Lang: {language}: {count}, {not_norm}, {ratio}")
    return ratio

data = concatenate_datasets([pub_data["train"], pub_data["validation"], test_data["test"]])
stats['norm'] = stats['lang'].apply(lambda x: norm_ratio(data, x))
stats["languages"] = stats["lang"].map(code2lang)

Lang: da: 20206, 18369, 9.09
Lang: de: 24948, 20609, 17.39
Lang: en: 73806, 68183, 7.62
Lang: es: 13824, 12790, 7.48
Lang: hr: 89052, 81781, 8.16
Lang: id: 48716, 25591, 47.47
Lang: iden: 23124, 19902, 13.93
Lang: it: 14641, 13614, 7.01
Lang: ja: 95416, 88713, 7.03
Lang: nl: 21657, 15412, 28.84
Lang: sl: 75276, 64038, 14.93
Lang: sr: 91738, 84512, 7.88
Lang: th: 200915, 192902, 3.99
Lang: tr: 8082, 5105, 36.83
Lang: trde: 16508, 12284, 25.59
Lang: vi: 128685, 108119, 15.98
Lang: ko: 16577, 15327, 7.54


In [18]:
subset = stats[stats["lang"].isin(newlangs)]
subset.sort_values("languages")[['languages', 'total', 'norm', 'mfr']]

,languages,total,norm,mfr
5,Indonesian,48716.0,47.47,59.75
8,Japanese,95416.0,7.03,6.32
16,Korean,16577.0,7.54,6.35
12,Thai,200915.0,3.99,42.77
15,Vietnamese,128685.0,15.98,75.77


In [19]:
stats[~stats["lang"].isin(newlangs)].sort_values("languages")[['languages', 'total', 'norm', 'mfr']]

,languages,total,norm,mfr
4,Croatian,89052.0,8.16,41.53
0,Danish,20206.0,9.09,49.68
9,Dutch,21657.0,28.84,39.39
2,English,73806.0,7.62,66.57
1,German,24948.0,17.39,34.35
6,Indonesian-English,23124.0,13.93,61.51
7,Italian,14641.0,7.01,16.83
11,Serbian,91738.0,7.88,45.19
10,Slovenian,75276.0,14.93,58.70
3,Spanish,13824.0,7.48,25.57
